**BONUS!**
Algunos ejercicios para seguir practicando:

1. Las patinetas que salgan menos que $68
2. Las patinetas que en su nombre tengan un numero mayor a 3
3. Traer cualquier texto de la pagina que tenga la palabra descuento u oferta.
4. Generar un archivo .csv con dos columnas: Una conteniendo el nombre del cliente y otra su testimonio.

In [ ]:
from bs4 import BeautifulSoup
from typing import List, Optional
import requests
import csv
import re
import pandas as pd

In [2]:
URL = 'https://scrapepark.org/courses/spanish/'
HEADERS = {
    'User-agent' : "Mozilla/5.0"
}

def get_soup(url:str, header:dict)  -> Optional[BeautifulSoup]:
    """Obtiene el objeto BeautifulSoup de una URL."""
    try:
        response = requests.get(url, headers=header, timeout=10)
        response.raise_for_status # Lanza error si no es 200
        
        return BeautifulSoup(response.text, "lxml")
    except requests.exceptions.RequestException as e:
        print(f"Error crítico de red: {e}")
        return None

soup = get_soup(URL, HEADERS)

### 1) Obtener los skate con precio menor a $68

In [3]:
def extract_skateboards(soup: BeautifulSoup) -> List:
    """Extrae productos usando contenedores para mayor robustez."""
    products_data = []
    
    # Buscamos específicamente en la sección de productos
    # Cada producto está dentro de un div con clase 'box'
    product_boxes = soup.select(".product-section .box")
    
    for box in product_boxes:
        title_tag = box.find("h5")
        price_tag = box.find("h6")
        
        if title_tag and price_tag:
            title = title_tag.get_text(strip=True)
            raw_price = price_tag.get_text(strip=True).replace("$","")
        
        try:
            price = float(raw_price)
            products_data.append({
                "product": title,
                "price": price
            })
        
        except ValueError:
            continue
        
    return products_data


skateboards = extract_skateboards(soup)

skateboards_df = pd.DataFrame(skateboards)

# Patinetas que tengan un precio menor a $68

cheap_skateboards = skateboards_df[ skateboards_df["price"] < 68 ].sort_values(by="price", ascending=False)
cheap_skateboards

,product,price
5,Patineta Nueva 6,58.0
9,Patineta Usada 10,54.0
7,Patineta Nueva 8,35.0


### 2) Las patinetas que en su nombre tengan un número mayor a 3

In [4]:
leaked_skateboards = skateboards_df[ skateboards_df["product"].str.contains("3") ]
leaked_skateboards

,product,price
2,Patineta Nueva 3,68.0


### 3) Traer cualquier texto de la pagina que tenga la palabra descuento u oferta.

In [46]:
texts = soup.find_all(string=re.compile("oferta|descuento", re.I))


for text in texts:
    print(f"{text.parent.name} -> {text.strip()}")

span -> Descuentos 20% Off
p -> Aprovechá nuestras ofertas.
span -> Descuentos 20% Off
p -> Aprovechá nuestras ofertas.
span -> Descuentos 20% Off
p -> Aprovechá nuestras ofertas.
h3 -> Suscríbete para obtener descuentos y ofertas


### 4) Generar un archivo .csv con dos columnas: Una conteniendo el nombre del cliente y otra su testimonio.

In [78]:
def get_customer_reviews() -> List:
    customer_reviews = []
    
    try:
        section_reviews = soup.select("#testimonios  .detail-box")

        for tag_customer_reviews in section_reviews:
            customer = tag_customer_reviews.find(class_="cliente-nombre")
            review = tag_customer_reviews.find(class_="cliente-comentario")
            
            if (customer and review):
                customer_reviews.append({
                    "Customer" : customer.get_text(strip=True),
                    "Review" : review .get_text(strip=True)
                    })
                
    except Exception as e:
        print(f"Error: ", e) 

    
    return customer_reviews

def convert_to_csv(data:List, name:str) -> None:
    
    try:
        with open(f"{name}.csv", "+w", newline='', encoding='utf-8') as f:
            # Obtener los header
            headers = data[0].keys()
            
            # utilizamos DictWriter dado que la estrucutra interna son diccionarios
            writer = csv.DictWriter(f, fieldnames=headers)
            
            # escribimos el encabezado de cada columna y sus filas
            writer.writeheader()
            writer.writerows(data)
            
            
    except Exception as e:
        print(f"Error: ", e)
        
        
data = get_customer_reviews()

convert_to_csv(data, "customer_reviews")

In [81]:
pd.read_csv("customer_reviews.csv")


,Customer,Review
0,Cliente 1,Los productos me encantaron y los precios son ...
1,Cliente 2,¡La calidad y variedad de patinetas es impresi...
2,Cliente 3,Estoy muy conforme. Hay muchas patinetas y los...
